<a href="https://colab.research.google.com/github/ZainabFatima-hzf/ML-flyRank/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZainabFatima-hzf/ML-flyRank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Primary method: Logistic Regression.** My question is exactly the shape the
`training-honest-models` skill maps to Logistic Regression first: a yes/no outcome
(`declined_next_28d`) with an observed label, evaluated by ranking (precision@K) rather than plain
accuracy. Logistic Regression gives me a probability per page (needed for ranking), and its
coefficients are directly readable — I can point at a number and say what it means, which matters
because the whole point of this lane is a decision a human acts on, not a black box.

**Second method: Random Forest — but only to test whether it's worth it.** The skill's table says
"readable → stronger," so I'm training a Random Forest on the exact same split and features, purely
to answer: does the added complexity actually beat Logistic Regression on *my* metric (precision@K),
or does it just look fancier? If it doesn't clearly win, I'm keeping Logistic Regression as the real
answer — "does not reward complexity alone" is a grading line, but it's also just good practice.

**Why not Gradient Boosting:** the menu says "where safe," and a handful of correlated,
modest-cardinality prior-window features on ~300K rows doesn't need it — it's the kind of case where
boosting would add variance and tuning surface without a clear reason to expect a real gain over a
Random Forest. I'm not ruling it out for the capstone, just not reaching for it in Week 5 without
evidence Random Forest already fell short.

**Features used (identical honesty rules as Weeks 3-4 — only what's knowable strictly before
`decision_date`):** `prior_28d_impressions`, `prior_28d_clicks`, `ctr` (clicks/impressions),
`prior_28d_avg_position`, `prior_28d_sessions`, `ga4_available_in_prior_window`,
`content_age_days`. **`days_since_update` is deliberately excluded** — Week 4 showed it's only
knowable for ~12% of rows at this decision date (the rest reflect an update *after* the decision
date, per `dim_content` only storing the latest update) and even reversed direction on that thin
slice. Including it here would repeat a mistake I already caught, not add a real signal.

In [1]:
# --- Self-contained setup: same honest pipeline as Weeks 3-4 (this notebook doesn't depend on
# earlier notebooks having run in this session). Needs HF_TOKEN in Colab Secrets.

!pip install -q duckdb huggingface_hub scikit-learn

import duckdb, os
import pandas as pd
import numpy as np
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql(f"""
    CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{os.environ["HF_TOKEN"]}')
""")

REPO = "hf://datasets/FlyRank/internship-warehouse"

# Only pull the 3 months we actually need (Feb, Mar, Apr 2026) -- not the whole 18-month panel.
DAILY = [
    f"{REPO}/fact_content_daily_performance/month=2026-02/*.parquet",
    f"{REPO}/fact_content_daily_performance/month=2026-03/*.parquet",
    f"{REPO}/fact_content_daily_performance/month=2026-04/*.parquet",
]
DIM_CONTENT = f"{REPO}/dim_content.parquet"

DECISION_DATE = "2026-03-15"
PRIOR_START   = "2026-02-15"   # 28 days before decision_date
NEXT_END      = "2026-04-12"   # 28 days after decision_date

# --- Prior-window features (identical construction to Weeks 3-4) ---
features = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id,
        SUM(gsc_impressions) AS prior_28d_impressions,
        SUM(gsc_clicks)      AS prior_28d_clicks,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS prior_28d_avg_position,
        SUM(ga4_sessions) FILTER (WHERE ga4_data_available IS TRUE) AS prior_28d_sessions,
        BOOL_OR(ga4_data_available) AS ga4_available_in_prior_window
    FROM read_parquet({DAILY})
    WHERE report_date >= '{PRIOR_START}' AND report_date < '{DECISION_DATE}'
    GROUP BY content_hash_id, client_hash_id
""").df()

# --- Forward-looking label (identical construction to Week 3) ---
label_frame = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_impressions) AS next_28d_impressions
    FROM read_parquet({DAILY})
    WHERE report_date >= '{DECISION_DATE}' AND report_date < '{NEXT_END}'
    GROUP BY content_hash_id, client_hash_id
""").df()

# --- Content metadata (age at decision date; days_since_update pulled but NOT used as a feature) ---
content_meta = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id,
        DATE '{DECISION_DATE}' - content_created_date AS content_age_days,
        DATE '{DECISION_DATE}' - content_updated_date AS days_since_update
    FROM read_parquet('{DIM_CONTENT}')
    WHERE is_deleted = FALSE
""").df()

demo = (
    features
    .merge(label_frame, on=["content_hash_id", "client_hash_id"], how="inner")
    .merge(content_meta, on=["content_hash_id", "client_hash_id"], how="left")
)
demo["declined_next_28d"] = (demo["next_28d_impressions"] < demo["prior_28d_impressions"]).astype(int)
demo["ctr"] = demo["prior_28d_clicks"] / demo["prior_28d_impressions"].replace(0, np.nan)

print("demo shape:", demo.shape)
print("label balance:")
print(demo["declined_next_28d"].value_counts(normalize=True).round(3))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

demo shape: (319584, 12)
label balance:
declined_next_28d
0    0.748
1    0.252
Name: proportion, dtype: float64


## 2. Split design

**Grouped by client, not random, and not touching future data.** I'm using a single 80/20
**GroupShuffleSplit on `client_hash_id`**, so every page belonging to a given client lands entirely
in train or entirely in test — never split across both. Two reasons this matters:

1. **Leakage risk between rows of the same client.** Pages from the same client tend to share
   templates, CMS quirks, and traffic patterns. A random row-level split would let the model partly
   memorize "what this client's pages look like" rather than learning a pattern that generalizes to
   a client it's never seen — exactly the failure mode the `flyrank-data` skill's uneven-panel
   warning points at.
2. **The real decision this supports is cross-client.** FlyRank reviews pages across many clients
   with one shared rule/model, not a separate model per client — so the honest test is "does this
   generalize to a client not in training," not "does it memorize clients it's already seen."

I am **not** doing a time-based split here on top of the client split, because the forward-window
label (`declined_next_28d`) already builds the future/past separation into every single row — the
leakage risk this lane cares about is "did the model see this row's own future," which the label
construction already prevents, not "did the model see a later calendar date than test."

In [2]:
from sklearn.model_selection import GroupShuffleSplit

FEATURE_COLS = [
    "prior_28d_impressions", "prior_28d_clicks", "ctr",
    "prior_28d_avg_position", "prior_28d_sessions",
    "ga4_available_in_prior_window", "content_age_days",
]

model_df = demo.dropna(subset=["prior_28d_avg_position", "content_age_days"]).copy()
# ga4-derived fields are legitimately missing for most rows (only ~4% GA4 coverage, per Week 3) --
# impute 0 sessions where GA4 wasn't available, and keep the availability flag as its own signal
model_df["prior_28d_sessions"] = model_df["prior_28d_sessions"].fillna(0)
model_df["ga4_available_in_prior_window"] = model_df["ga4_available_in_prior_window"].fillna(False).astype(int)
model_df["ctr"] = model_df["ctr"].fillna(0)  # 0 impressions -> 0 ctr is the honest value, not "unknown"

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(model_df, groups=model_df["client_hash_id"]))
train_df, test_df = model_df.iloc[train_idx].copy(), model_df.iloc[test_idx].copy()

overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
print(f"Train: {len(train_df)} rows, {train_df['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test_df)} rows, {test_df['client_hash_id'].nunique()} clients")
print(f"Client overlap between train and test (must be 0): {len(overlap)}")
print(f"Test-set base rate: {test_df['declined_next_28d'].mean():.3f}")


Train: 144848 rows, 34 clients
Test:  14312 rows, 9 clients
Client overlap between train and test (must be 0): 0
Test-set base rate: 0.428


## 3. Train + compare vs my baseline

Same features, same forward-looking label, same grouped-by-client split design as Week 4's
recomputed baseline — but because the test fold only has 9 clients, a single split is too noisy to
trust on its own (I checked: the test-set base rate alone swings from 0.387 to 0.631 just by
changing the random seed, with nothing else different). So the table below is averaged across 5
different client-grouped splits (seeds 0, 1, 2, 42, 99), reporting mean ± std rather than one
number.

| method | base_rate | precision@50 | precision@200 |
|---|---|---|---|
| Week-4 rule | 0.516 ± 0.104 | 0.456 ± 0.206 | 0.454 ± 0.200 |
| Logistic Regression | 0.516 ± 0.104 | 0.712 ± 0.125 | 0.674 ± 0.147 |
| Random Forest | 0.516 ± 0.104 | 0.676 ± 0.155 | 0.617 ± 0.128 |

**Logistic Regression wins, clearly and consistently.** It beats the Week-4 rule by roughly 0.25
at precision@50, across every split checked — not a one-off.

**Random Forest does not beat Logistic Regression.** It scores slightly lower on average and is
more volatile (higher std) across seeds. Per the `training-honest-models` guidance — add
complexity only when the comparison earns it — I'm keeping **Logistic Regression** as the model for
this lane, not Random Forest, even though it's the "simpler" option.

**The most surprising finding is the rule's own number.** The Week-4 rule averages *below* the base
rate at precision@50 (0.456 vs. 0.516) — worse than picking pages at random. The CTR-vs-position
signal that looked CONFIRMED in Week 4's isolated bucket check doesn't survive being turned into an
actual ranking rule with combined gates and thresholds. That gap between "a signal holds up in
isolation" and "a rule built from it beats chance" is the real lesson of this comparison, not the
headline model score.

In [8]:
results = []
for seed in [0, 1, 2, 42, 99]:
    sp = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_idx, te_idx = next(sp.split(model_df, groups=model_df["client_hash_id"]))
    tr, te = model_df.iloc[tr_idx], model_df.iloc[te_idx].copy()

    Xtr, ytr = tr[FEATURE_COLS], tr["declined_next_28d"]
    Xte = te[FEATURE_COLS]

    sc = StandardScaler().fit(Xtr)
    lr = LogisticRegression(max_iter=1000, random_state=42).fit(sc.transform(Xtr), ytr)
    rf_s = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1).fit(Xtr, ytr)

    te["logreg_score"] = lr.predict_proba(sc.transform(Xte))[:, 1]
    te["rf_score"] = rf_s.predict_proba(Xte)[:, 1]
    ctr_gate = ((te["prior_28d_impressions"] >= 100) & (te["prior_28d_avg_position"].between(0.01, 20)) & (te["ctr"] < 0.02)).astype(int)
    te["rule_score"] = ctr_gate * te["prior_28d_impressions"]

    for name, col in [("Week-4 rule", "rule_score"), ("Logistic Regression", "logreg_score"), ("Random Forest", "rf_score")]:
        results.append({"seed": seed, "method": name, "base_rate": te["declined_next_28d"].mean(),
                         "precision@50": precision_at_k(te, col, 50), "precision@200": precision_at_k(te, col, 200)})

res_df = pd.DataFrame(results)
summary = res_df.groupby("method")[["base_rate", "precision@50", "precision@200"]].agg(["mean", "std"]).round(3)
print(summary)

                    base_rate        precision@50        precision@200       
                         mean    std         mean    std          mean    std
method                                                                       
Logistic Regression     0.516  0.104        0.712  0.125         0.674  0.147
Random Forest           0.516  0.104        0.676  0.155         0.617  0.128
Week-4 rule             0.516  0.104        0.456  0.206         0.454  0.200


## 4. Errors and interpretation

**What Logistic Regression leans on:** `prior_28d_clicks` has the largest standardized coefficient
(-0.51), followed by `prior_28d_sessions` (+0.28) and `prior_28d_impressions` (+0.25). The negative
sign on clicks and positive sign on sessions/impressions is a little counterintuitive at first read
— worth treating as a correlation among related volume signals rather than three independent causal
effects, since all three move together for a given page.

**What Random Forest leans on (permutation importance):** `content_age_days` dominates (0.091 —
roughly 4x the next feature), with `ctr` and `prior_28d_avg_position` well behind. I checked whether
this is a real content signal or just a stand-in for "which client is this": grouping by client
shows decline rate climbing fairly steadily from near-0 in brand-new clients (avg content age ~5-10
days) up to 0.6-0.8 in the oldest ones (avg age 150-250+ days), consistently across many different
clients rather than one or two outliers driving it. That's a real between-client pattern — older
client cohorts decline more — but I can't yet say whether it also holds *within* one client's own
catalog (their older pages vs. their newer ones), since this check only compares client averages.
That's a fair "what would make this wrong": if it turns out to be purely a between-client effect,
`content_age_days` is partly acting as a client fingerprint, not a portable content signal — worth
checking with a within-client comparison before leaning on it for the capstone.

**3 concrete wrong cases** (from the seed=42 split, one representative run among the several
checked above — individual wrong cases will differ split to split, but the pattern doesn't):
these are pages with moderate impressions (1,400-2,700) and *extremely* low CTR (0.00-0.08%,
essentially zero clicks despite thousands of impressions). The model scored them as likely to
decline, and they didn't. The most plausible explanation, and the honest caveat: a CTR this close
to zero is exactly the pattern you'd also see if click tracking failed for that page, rather than
genuine total disinterest — the model has no way to tell "broken tracking" from "real problem" apart
from the number itself, and neither can I without checking GSC directly.

**Overall takeaway:** Logistic Regression gives a real, repeatable improvement over the hand-built
rule, but the honest story here is as much about what *didn't* hold up (the rule underperforming
chance, Random Forest not earning its complexity) as what did — and that's the useful result to
carry into the capstone, not just the winning number.

In [4]:
from sklearn.inspection import permutation_importance

# --- What Logistic Regression leans on (coefficients on standardized features -> comparable) ---
coef_table = pd.DataFrame({
    "feature": FEATURE_COLS,
    "coefficient": logreg.coef_[0],
}).sort_values("coefficient", key=abs, ascending=False)
print("Logistic Regression coefficients (standardized, sorted by |effect|):")
print(coef_table.to_string(index=False))
print()

# --- Permutation importance on whichever model wins the comparison table above ---
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, scoring="roc_auc")
perm_table = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)
print("Random Forest permutation importance (drop in ROC-AUC when shuffled):")
print(perm_table.round(4).to_string(index=False))
print()
print("Sanity check: does the top feature make sense, or is it suspiciously perfect?")
print("If ctr or position dominates, that lines up with Week 4's CONFIRMED signal check --")
print("consistent, not a red flag. A near-1.0 importance on a single column would be the red flag.")

# --- 3 concrete wrong cases, using whichever model scored best on precision@50 above ---
best_col = comparison.sort_values("precision@50", ascending=False).iloc[0]["method"]
score_col = {"Week-4 rule": "rule_score", "Logistic Regression": "logreg_score", "Random Forest": "rf_score"}[best_col]

top50 = test_df.sort_values(score_col, ascending=False).head(50)
wrong = top50[top50["declined_next_28d"] == 0].head(3)
print()
print(f"3 concrete wrong cases from the top 50 by {best_col} ({score_col}):")
print(wrong[["content_hash_id", score_col, "prior_28d_impressions", "ctr", "prior_28d_avg_position"]].to_string(index=False))


Logistic Regression coefficients (standardized, sorted by |effect|):
                      feature  coefficient
             prior_28d_clicks    -0.507811
           prior_28d_sessions     0.276965
        prior_28d_impressions     0.250337
       prior_28d_avg_position    -0.236203
ga4_available_in_prior_window    -0.158798
             content_age_days     0.044400
                          ctr     0.014772

Random Forest permutation importance (drop in ROC-AUC when shuffled):
                      feature  importance_mean  importance_std
             content_age_days           0.0910          0.0025
                          ctr           0.0234          0.0009
       prior_28d_avg_position           0.0217          0.0023
        prior_28d_impressions           0.0131          0.0013
             prior_28d_clicks           0.0104          0.0007
ga4_available_in_prior_window           0.0042          0.0012
           prior_28d_sessions           0.0038          0.0009

Sanity chec

In [7]:
for seed in [0, 1, 2, 42, 99]:
    sp = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_idx, te_idx = next(sp.split(model_df, groups=model_df["client_hash_id"]))
    te = model_df.iloc[te_idx]
    print(f"seed={seed}: n_test_clients={te['client_hash_id'].nunique()}, base_rate={te['declined_next_28d'].mean():.3f}")

seed=0: n_test_clients=9, base_rate=0.631
seed=1: n_test_clients=9, base_rate=0.583
seed=2: n_test_clients=9, base_rate=0.551
seed=42: n_test_clients=9, base_rate=0.428
seed=99: n_test_clients=9, base_rate=0.387
